In [1]:
# ============================================================
# DRONECROWD PIPELINE — CrowdNet Adaptation  (v8)
# ============================================================
#
# Reference: Wen et al., "Detection, Tracking, and Counting
# Meets Drones in Crowds: A Benchmark", CVPR 2021.
#
# ============================================================
# EVALUATION PROTOCOL — matches official STNNet (mytrain.py/mytest.py)
# ============================================================
#
#   train_data  (82 seqs, 300 frames) → gradient updates, T=15 windows
#   val_data    (30 seqs,  12 frames) → checkpoint selection, T=11 windows
#   test_data   (30 seqs, 300 frames) → final evaluation only
#
#   Training: fixed 100 epochs, NO early stopping (STNNet protocol).
#   Best val-MAE checkpoint is saved throughout; reported at the end.
#
#   STNNet MAE note: they sum density across 4 FiveCrops of 960×540,
#   then divide by factor=100 for per-frame count. They evaluate on
#   9000 individual frames (30 seqs × 300 frames), one at a time.
#   Our evaluation: per-frame across all (B×T) frame predictions in
#   sliding windows over the same 30 test sequences. Metric is equivalent.
#
# ============================================================
# ARCHITECTURE — v8 CHANGES
# ============================================================
#
#  INPUT RESOLUTION: 448×448 (up from 224×224)
#    → stage1_density: (B, 24, 56×56)   fine spatial detail
#    → d_mid:          (B, 48, 28×28)   intermediate
#    → d_stage2:       (B, 576, 14×14)  semantic context
#    → density output: 56×56 (each cell = 34×34px in 1920px space)
#    Previous 224→28 gave 69px/cell; now 448→56 gives 34px/cell.
#    Halves the minimum resolvable crowd spacing.
#
#  THREE-SCALE FPN + ASPP-LITE density path (isolated, trainable):
#    d_stage1 (56×56) + d_mid (28→56) + d_stage2 (14→56) fused
#    → 4-branch ASPP with dilation 1,2,4,6 at 56×56 resolution
#    → density_project → (B,1,56,56) density map
#
#  CLASSIFICATION PATH: fully frozen (original backbone_stage1/2)
#    Task B safety: density training never touches classification weights.
#
# ============================================================
# CUMULATIVE FIXES
# ============================================================
#
# FIX 1  — Per-frame GT counts stored (not window mean).
#           MAE = mean |pred(t) - GT(t)| over every individual frame.
# FIX 2  — nn.ReLU() as final density head layer.
# FIX 3  — Per-frame count loss: L1(pred_per_frame, gt_per_frame).
# FIX 4  — Differential LR: aspp/fpn 3e-4, d_stage1 3e-5,
#           d_mid 3e-5, d_stage2 1e-5.
# FIX 5  — Task B uses best_dronecrowd_model.pth directly.
#           Classification weights are synthetic and frozen throughout.
# FIX 6  — Official STNNet val protocol: val_data (12 frames/seq).
# FIX 7  — Density head keys excluded from synthetic checkpoint load.
# FIX 8  — 3-scale FPN + ASPP-lite density head.
# FIX 9  — Patch-averaged Farneback flow for Task C.
# FIX 10 — Input resolution 448×448, density output 56×56.
# FIX 11 — Fixed 100-epoch training, no early stopping.
#
# ============================================================

import os
import re
import glob
import math
import random
from collections import defaultdict

import cv2
import numpy as np
import scipy.io
import scipy.ndimage
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 60)
print(f"DEVICE : {device}")
print("=" * 60)


# ============================================================
# GLOBAL CONFIG
# ============================================================

DRONECROWD_ROOT  = "dronecrowd"
PROCESSED_DIR    = "dronecrowd_processed"

IMG_W, IMG_H     = 1920, 1080
MODEL_SIZE       = 448   # matches preprocessed data (448×448 frames on disk)
DENSITY_RES      = 56    # 448/56=8 px per cell; same ratio as 224/28=8
DENSITY_SIGMA    = 8    # MODEL_SIZE/DENSITY_RES=8 always (224/28=448/56=8)
DENSITY_SCALE    = float((MODEL_SIZE / DENSITY_RES) ** 2)   # (448/56)^2 = 64.0

FRAMES_PER_SEQ   = 300
MIN_FRAMES       = 20
SEQ_LENGTH       = 15
STRIDE           = 5
WARMUP_FRAMES    = 3

EPOCHS           = 100   # fixed epochs, matching STNNet protocol (no early stopping)
BATCH_SIZE       = 4
LR               = 3e-4
WEIGHT_DECAY     = 5e-4
PATIENCE         = 15

COUNT_LOSS_WEIGHT = 0.05   # increased from 0.01 — stronger anti-overfitting
                           # anchor; pixel loss alone let train MAE reach 3.95
                           # while val MAE stayed 42-68 (severe overfit)

CROWDED_THRESH   = 150
CLASS_NAMES      = {0: "Normal", 1: "Bottleneck", 2: "Panic"}

# ── RUN MODE ─────────────────────────────────────────────────
MAX_SEQS_SMOKE    = None   # None = full run; 3 = smoke test
TASK_C_MAX_SEQS   = 5
TASK_C_MAX_FRAMES = 50

# ── VAL SPLIT FROM TRAIN (FIX 6: shuffled) ───────────────────
# ── VALIDATION PROTOCOL ───────────────────────────────────────
# Matches the OFFICIAL STNNet protocol verified from source
# (mytrain.py / mytest.py): train_data for gradients, val_data for
# every-epoch early-stopping signal (no gradients), test_data for
# final evaluation only. See DroneCrowdDataset docstring for the
# val_data short-window handling (12 frames/seq → 1 window/seq).


# ============================================================
# STEP 0 : FILENAME UTILITIES
# ============================================================

def parse_image_name(fname):
    m = re.match(r"(?:GT_)?img(\d{3})(\d{3})(?:\.jpg|\.mat)?$",
                 os.path.basename(fname))
    return (int(m.group(1)), int(m.group(2))) if m else (None, None)

def image_name(seq_id, frame_id): return f"img{seq_id:03d}{frame_id:03d}.jpg"
def gt_name(seq_id, frame_id):    return f"GT_img{seq_id:03d}{frame_id:03d}.mat"


# ============================================================
# STEP 1 : READ ONE .MAT ANNOTATION FILE
# ============================================================

def read_mat(mat_path):
    mat  = scipy.io.loadmat(mat_path)
    cell = mat["image_info"][0, 0]
    loc  = cell["location"][0, 0].astype(np.float32)
    return loc[:, :2], loc[:, 2].astype(np.int32), int(cell["number"][0, 0].item())


# ============================================================
# STEP 2 : BUILD DENSITY MAP
# ============================================================

def build_density_map(points):
    """(N,2) head centres 1920×1080 → (28,28) map, sum==N."""
    density_full = np.zeros((MODEL_SIZE, MODEL_SIZE), dtype=np.float32)
    sx = MODEL_SIZE / IMG_W;  sy = MODEL_SIZE / IMG_H
    for (x, y) in points:
        density_full[int(np.clip(y*sy,0,MODEL_SIZE-1)),
                     int(np.clip(x*sx,0,MODEL_SIZE-1))] += 1.0
    density_full  = scipy.ndimage.gaussian_filter(density_full, sigma=DENSITY_SIGMA)
    density_small = cv2.resize(density_full, (DENSITY_RES, DENSITY_RES),
                               interpolation=cv2.INTER_AREA)
    return (density_small * DENSITY_SCALE).astype(np.float32)


# ============================================================
# STEP 3 : DISCOVER SEQUENCES
# ============================================================

def discover_sequences(split_name, dronecrowd_root, min_frames=None):
    min_frames = MIN_FRAMES if min_frames is None else min_frames
    img_dir = os.path.join(dronecrowd_root, split_name, "images")
    if not os.path.exists(img_dir):
        print(f"  [{split_name}] images/ not found"); return {}
    all_files = os.listdir(img_dir)
    if not all_files:
        print(f"  [{split_name}] images/ is EMPTY"); return {}
    print(f"  [{split_name}] sample filenames: {sorted(all_files)[:4]}")
    seq_frames = defaultdict(list)
    for fname in all_files:
        sid, fid = parse_image_name(fname)
        if sid is not None: seq_frames[sid].append(fid)
    seq_dict = {sid: sorted(fids) for sid, fids in seq_frames.items()
                if len(fids) >= min_frames}
    print(f"  [{split_name}] found {len(seq_dict)} sequences "
          f"({sum(len(v) for v in seq_dict.values())} frames total)  "
          f"[min_frames={min_frames}]")
    return seq_dict


# ============================================================
# STEP 4 : PRE-PROCESS A SPLIT
# ============================================================

def _compute_and_save_flows(out_seq, frame_ids):
    """Compute missing flow files using existing frame PNGs."""
    frames_cache = []
    for t in range(len(frame_ids)):
        img = cv2.imread(os.path.join(out_seq, f"frame_{t:03d}.png"),
                         cv2.IMREAD_GRAYSCALE)
        frames_cache.append(img if img is not None
                            else np.zeros((MODEL_SIZE,MODEL_SIZE),dtype=np.uint8))
    saved = 0
    for t in range(len(frames_cache)-1):
        flow_path = os.path.join(out_seq, f"flow_{t:03d}.npy")
        if not os.path.exists(flow_path):
            flow = cv2.calcOpticalFlowFarneback(
                frames_cache[t], frames_cache[t+1], None,
                pyr_scale=0.5,levels=3,winsize=15,
                iterations=3,poly_n=5,poly_sigma=1.2,flags=0)
            np.save(flow_path,
                    np.clip(flow/8.0,-1.0,1.0).astype(np.float16))
            saved += 1
    return saved


def preprocess_split(split_name, dronecrowd_root, out_root,
                     max_seqs=MAX_SEQS_SMOKE, min_frames_override=None):
    img_dir  = os.path.join(dronecrowd_root, split_name, "images")
    gt_dir   = os.path.join(dronecrowd_root, split_name, "ground_truth")
    seq_dict = discover_sequences(split_name, dronecrowd_root,
                                   min_frames=min_frames_override)
    if not seq_dict: return

    seq_ids = sorted(seq_dict.keys())
    if max_seqs is not None:
        seq_ids = seq_ids[:max_seqs]
        print(f"  SMOKE TEST: {len(seq_ids)} seqs ({seq_ids[0]}..{seq_ids[-1]})")

    for seq_id in tqdm(seq_ids, desc=f"  preprocess {split_name}"):
        frame_ids        = seq_dict[seq_id]
        out_seq          = os.path.join(out_root, split_name, f"seq{seq_id:03d}")
        n_flows_expected = len(frame_ids) - 1   # 299 for 300-frame seq
        existing         = glob.glob(os.path.join(out_seq,"frame_*.png"))
        flow_existing    = glob.glob(os.path.join(out_seq,"flow_*.npy"))
        traj_exists      = os.path.exists(os.path.join(out_seq,"trajectories.npy"))
        frames_ok        = len(existing) == len(frame_ids)
        flows_ok         = len(flow_existing) == n_flows_expected

        # Case 1: fully done
        if frames_ok and flows_ok and traj_exists:
            print(f"    seq{seq_id:03d}: done "
                  f"({len(frame_ids)} frames, {n_flows_expected} flows ✓)")
            continue

        # Case 2: frames + traj done, only flows missing
        if frames_ok and traj_exists and not flows_ok:
            n = _compute_and_save_flows(out_seq, frame_ids)
            print(f"    seq{seq_id:03d}: added {n} missing flows")
            continue

        # Case 3: frames done, trajectories missing
        if frames_ok and not traj_exists:
            print(f"    seq{seq_id:03d}: regenerating trajectories"
                  + (" + flows" if not flows_ok else ""))
            traj_rows = []; sx=MODEL_SIZE/IMG_W; sy=MODEL_SIZE/IMG_H
            for t, fid in enumerate(frame_ids):
                mat_path = os.path.join(gt_dir, gt_name(seq_id, fid))
                if os.path.exists(mat_path):
                    points, track_ids, _ = read_mat(mat_path)
                    for i,(x,y) in enumerate(points):
                        traj_rows.append([float(t),float(x*sx),
                                          float(y*sy),float(track_ids[i])])
            if traj_rows:
                np.save(os.path.join(out_seq,"trajectories.npy"),
                        np.array(traj_rows,dtype=np.float32))
            if not flows_ok:
                _compute_and_save_flows(out_seq, frame_ids)
            continue

        # Case 4: fresh processing
        os.makedirs(out_seq, exist_ok=True)
        gt_counts=[]; traj_rows=[]; frames_cache=[]
        sx=MODEL_SIZE/IMG_W; sy=MODEL_SIZE/IMG_H
        for t, fid in enumerate(frame_ids):
            img = cv2.imread(os.path.join(img_dir,image_name(seq_id,fid)),
                             cv2.IMREAD_GRAYSCALE)
            img = (cv2.resize(img,(MODEL_SIZE,MODEL_SIZE)) if img is not None
                   else np.zeros((MODEL_SIZE,MODEL_SIZE),dtype=np.uint8))
            cv2.imwrite(os.path.join(out_seq,f"frame_{t:03d}.png"), img)
            frames_cache.append(img)

            mat_path = os.path.join(gt_dir, gt_name(seq_id,fid))
            if os.path.exists(mat_path):
                points, track_ids, count = read_mat(mat_path)
            else:
                points=np.zeros((0,2),dtype=np.float32)
                track_ids=np.zeros(0,dtype=np.int32); count=0
            gt_counts.append(float(count))
            np.save(os.path.join(out_seq,f"density_{t:03d}.npy"),
                    build_density_map(points))
            for i,(x,y) in enumerate(points):
                traj_rows.append([float(t),float(x*sx),
                                  float(y*sy),float(track_ids[i])])
            if t > 0:
                flow = cv2.calcOpticalFlowFarneback(
                    frames_cache[t-1],frames_cache[t],None,
                    pyr_scale=0.5,levels=3,winsize=15,
                    iterations=3,poly_n=5,poly_sigma=1.2,flags=0)
                np.save(os.path.join(out_seq,f"flow_{t-1:03d}.npy"),
                        np.clip(flow/8.0,-1.0,1.0).astype(np.float16))

        np.save(os.path.join(out_seq,"gt_counts.npy"),
                np.array(gt_counts,dtype=np.float32))
        if traj_rows:
            np.save(os.path.join(out_seq,"trajectories.npy"),
                    np.array(traj_rows,dtype=np.float32))
        d0=np.load(os.path.join(out_seq,"density_000.npy"))
        print(f"    seq{seq_id:03d}: {len(frame_ids)} frames | "
              f"GT0={gt_counts[0]:.0f} | density_sum={d0.sum():.1f}")


def preprocess_all(dronecrowd_root, out_root):
    """
    Processes train_data, val_data, and test_data.

    val_data has only VAL_DATA_FRAMES=12 frames per sequence, well
    below MIN_FRAMES=20 used for train/test. It is preprocessed with
    its own minimum threshold so its short sequences are not silently
    dropped by discover_sequences().
    """
    os.makedirs(out_root, exist_ok=True)

    for split in ["train_data","test_data"]:
        if not os.path.exists(os.path.join(dronecrowd_root,split)):
            raise FileNotFoundError(f"Missing: {dronecrowd_root}/{split}")
        print(f"\n{'='*50}\nPre-processing: {split}\n{'='*50}")
        preprocess_split(split, dronecrowd_root, out_root)

    val_dir = os.path.join(dronecrowd_root, "val_data")
    if os.path.exists(val_dir):
        print(f"\n{'='*50}\nPre-processing: val_data\n{'='*50}")
        preprocess_split("val_data", dronecrowd_root, out_root,
                         min_frames_override=DroneCrowdDataset.VAL_DATA_FRAMES)
    else:
        print(f"\nWARNING: val_data/ not found at {dronecrowd_root}/val_data")
        print(f"  Validation requires the official val_data folder.")
        print(f"  Without it, training cannot proceed with the STNNet-matched protocol.")

    print("\nPre-processing complete.")


# ============================================================
# SANITY CHECK
# ============================================================

def sanity_check_density(processed_root, n_frames=8):
    all_pass = True
    for split in ["train_data","val_data","test_data"]:
        split_root = os.path.join(processed_root, split)
        if not os.path.exists(split_root): continue
        seqs = sorted([s for s in os.listdir(split_root)
                       if os.path.isdir(os.path.join(split_root,s))])
        if not seqs: continue
        seq_path  = os.path.join(split_root, seqs[0])
        gt_counts = np.load(os.path.join(seq_path,"gt_counts.npy"))
        print(f"\n{'='*60}")
        print(f"DENSITY SANITY CHECK [{split}] seq:{seqs[0]}")
        print(f"{'Frame':>6}  {'GT':>8}  {'MapSum':>8}  {'Diff':>8}  Status")
        print("-"*50)
        for t in range(min(n_frames,len(gt_counts))):
            d_path = os.path.join(seq_path,f"density_{t:03d}.npy")
            if not os.path.exists(d_path): continue
            d=np.load(d_path); gt=float(gt_counts[t]); s=float(d.sum())
            ok=abs(s-gt)<max(2.0,gt*0.02)
            if not ok: all_pass=False
            print(f"{t:>6}  {gt:>8.1f}  {s:>8.2f}  {s-gt:>+8.2f}  "
                  f"{'✓' if ok else '✗'}")
        print("="*60)
    print(f"\nSanity: {'PASSED ✓' if all_pass else 'FAILED ✗'}")
    if not all_pass:
        raise RuntimeError("Density sanity failed.")


# ============================================================
# CELL 2 : DATASET
# ============================================================

def augment_frames(frames_np, aug_prob=0.5):
    if np.random.rand() > aug_prob: return frames_np
    mode = np.random.choice(
        ["gaussian","blur","brightness","salt_pepper","fog","frame_dropout","none"],
        p=[0.17,0.14,0.14,0.18,0.14,0.07,0.16])
    if mode == "none": return frames_np
    aug = []
    if mode == "gaussian":
        s=np.random.uniform(5,20)
        for f in frames_np:
            aug.append(np.clip(f.astype(np.float32)+np.random.normal(0,s,f.shape),0,255).astype(np.uint8))
    elif mode == "blur":
        k=np.random.choice([3,5,7,9])
        for f in frames_np: aug.append(cv2.GaussianBlur(f,(k,k),0))
    elif mode == "brightness":
        fac=np.random.uniform(0.5,1.6)
        for f in frames_np:
            aug.append(np.clip(f.astype(np.float32)*fac,0,255).astype(np.uint8))
    elif mode == "salt_pepper":
        p=np.random.uniform(0.005,0.05)
        for f in frames_np:
            a=f.copy(); mask=np.random.rand(*f.shape)
            a[mask<p/2]=0; a[(mask>=p/2)&(mask<p)]=255; aug.append(a)
    elif mode == "fog":
        alpha=np.random.uniform(0.15,0.50)
        for f in frames_np:
            aug.append(np.clip((1-alpha)*f.astype(np.float32)+alpha*128,0,255).astype(np.uint8))
    elif mode == "frame_dropout":
        aug=[f.copy() for f in frames_np]
        for idx in np.random.choice(len(aug),
                                     size=np.random.choice([1,2],p=[0.8,0.2]),
                                     replace=False):
            aug[idx][:] = (0 if np.random.rand()<0.5
                           else (aug[idx-1].copy() if idx>0 else aug[idx]))
        return aug
    return aug


def augment_flow(ft, training=True):
    if not training: return ft
    ft = ft * np.random.uniform(0.8,1.2)
    if np.random.rand() < 0.5:
        ft=torch.flip(ft,dims=[3]); ft[:,0,:,:]=-ft[:,0,:,:]
    s=(np.random.uniform(0.005,0.02) if np.random.rand()>0.1
       else np.random.uniform(0.02,0.05))
    return torch.clamp(ft+torch.randn_like(ft)*s,-1,1)


class DroneCrowdDataset(Dataset):
    """
    split: "train" | "val" | "test_data"

    Matches the OFFICIAL STNNet protocol, verified directly from
    the released mytrain.py / mytest.py source:
        train_data → training (gradient updates)
        val_data   → early-stopping signal every epoch (torch.no_grad())
        test_data  → final evaluation only, completely separate

    "train" reads from processed/train_data/ — ALL 82 sequences,
        windowed at seq_length=SEQ_LENGTH (15) with sliding stride.
    "val" reads from processed/val_data/ — 30 sequences with only
        12 frames each. Cannot form a SEQ_LENGTH=15 window (needs
        >=19 frames minimum). Instead uses ONE window per sequence
        spanning ALL 12 frames (11 flow-steps, no warmup skip,
        no stride loop) — this preserves the ConvLSTM's temporal
        structure for that window rather than degrading to T=1,
        while still fitting in the available 12 frames.
    "test_data" reads from processed/test_data/ — 30 sequences with
        the full 300 frames each, windowed normally for Task A/B/C.

    NOTE on val_data/test_data frame overlap: per direct inspection,
    val_data's 12 frames per sequence are the SAME first 12 frames
    also present in test_data's 300 frames for the same sequence ID.
    This mild (4% of frames) overlap is inherent to the official
    VisDrone download and is present in STNNet's own reported
    numbers too (their validate() never backprops on val_data, only
    uses it for checkpoint selection) — we replicate this exactly
    for a fair, literature-comparable evaluation.

    FIX 1: density target is the MEAN GT count over all frames in
    the window (not just the last frame), aligning training/eval
    and giving the count loss a stable target.
    """

    VAL_DATA_FRAMES = 12   # frames available per sequence in val_data
    VAL_SEQ_LENGTH  = 11   # 12 frames → 11 consecutive flow steps

    def __init__(self, processed_root, split,
                 seq_length=SEQ_LENGTH, stride=STRIDE,
                 training=False, aug_prob=0.5):
        self.training=training; self.aug_prob=aug_prob; self.windows=[]

        if split == "val":
            disk_split = "val_data"
            self.seq_length = self.VAL_SEQ_LENGTH
        elif split == "train":
            disk_split = "train_data"
            self.seq_length = seq_length
        else:
            disk_split = split   # "test_data"
            self.seq_length = seq_length

        split_root = os.path.join(processed_root, disk_split)
        if not os.path.exists(split_root):
            raise FileNotFoundError(
                f"{split_root} not found — run preprocess_all() first.\n"
                f"If split='val', make sure val_data/ has been preprocessed "
                f"(see preprocess_val_data()).")

        seq_names = sorted([s for s in os.listdir(split_root)
                            if os.path.isdir(os.path.join(split_root,s))])

        if split == "val":
            # ONE window per sequence, using ALL available frames.
            # No warmup skip, no stride loop — every val sequence
            # contributes exactly one window, start=0.
            for seq_name in seq_names:
                seq_path=os.path.join(split_root,seq_name)
                n_frames=len(glob.glob(os.path.join(seq_path,"frame_*.png")))
                if n_frames < self.seq_length+1:
                    print(f"    WARNING: {seq_name} has only {n_frames} frames, "
                          f"need >= {self.seq_length+1} — skipping")
                    continue
                gt_path=os.path.join(seq_path,"gt_counts.npy")
                gt_arr=(np.load(gt_path) if os.path.exists(gt_path)
                        else np.zeros(n_frames,dtype=np.float32))
                window_gt=gt_arr[:self.seq_length]
                self.windows.append({
                    "seq_path": seq_path, "start": 0,
                    # Store per-frame counts (T,) not mean — fixes metric bug
                    "gt_counts_per_frame": window_gt.tolist() if len(window_gt)>0
                                           else [0.0]*self.seq_length
                })
            print(f"  val: {len(seq_names)} seqs from val_data/ "
                  f"(1 window/seq, seq_length={self.seq_length})")

        else:
            for seq_name in seq_names:
                seq_path=os.path.join(split_root,seq_name)
                n_frames=len(glob.glob(os.path.join(seq_path,"frame_*.png")))
                if n_frames < self.seq_length+1: continue
                gt_path=os.path.join(seq_path,"gt_counts.npy")
                gt_arr=(np.load(gt_path) if os.path.exists(gt_path)
                        else np.zeros(n_frames,dtype=np.float32))
                for start in range(WARMUP_FRAMES, n_frames-self.seq_length, stride):
                    window_gt = gt_arr[start:start+self.seq_length]
                    self.windows.append({
                        "seq_path": seq_path, "start": start,
                        # Per-frame GT counts (T,) — correct per-frame MAE metric
                        "gt_counts_per_frame": window_gt.tolist() if len(window_gt)>0
                                               else [0.0]*self.seq_length
                    })
            print(f"  {split}: {len(seq_names)} seqs from {disk_split}/")

        print(f"DroneCrowd [{split}] — {len(seq_names)} seqs, "
              f"{len(self.windows)} windows")

    def __len__(self): return len(self.windows)

    def __getitem__(self, idx):
        w=self.windows[idx]; seq_path=w["seq_path"]
        start=w["start"]
        # Per-frame GT counts (T,) — correct DroneCrowd per-frame MAE metric
        gt_per_frame = torch.tensor(w["gt_counts_per_frame"], dtype=torch.float32)

        raw=[]
        for t in range(start, start+self.seq_length+1):
            img=Image.open(
                os.path.join(seq_path,f"frame_{t:03d}.png")
            ).convert("L").resize((MODEL_SIZE,MODEL_SIZE))
            raw.append(np.array(img))
        if self.training:
            raw=augment_frames(raw, self.aug_prob)

        flows=[]
        for t in range(self.seq_length):
            flow_path=os.path.join(seq_path,f"flow_{start+t:03d}.npy")
            if os.path.exists(flow_path):
                flow=np.load(flow_path).astype(np.float32)
                if self.training and self.aug_prob>0 and np.random.rand()<0.3:
                    fr=cv2.calcOpticalFlowFarneback(
                        raw[t],raw[t+1],None,pyr_scale=0.5,levels=3,
                        winsize=15,iterations=3,poly_n=5,poly_sigma=1.2,flags=0)
                    flow=np.clip(fr/8.0,-1.0,1.0)
                    flow=np.stack([cv2.medianBlur(flow[:,:,0],3),
                                   cv2.medianBlur(flow[:,:,1],3)],axis=2)
            else:
                fr=cv2.calcOpticalFlowFarneback(
                    raw[t],raw[t+1],None,pyr_scale=0.5,levels=3,
                    winsize=15,iterations=3,poly_n=5,poly_sigma=1.2,flags=0)
                flow=np.clip(fr/8.0,-1.0,1.0)
                if self.training and np.random.rand()<0.3:
                    flow=np.stack([cv2.medianBlur(flow[:,:,0],3),
                                   cv2.medianBlur(flow[:,:,1],3)],axis=2)
            flows.append(torch.tensor(flow,dtype=torch.float32).permute(2,0,1))

        frames=[
            torch.tensor(raw[t].astype(np.float32)/255.0,
                         dtype=torch.float32).unsqueeze(0)
            for t in range(self.seq_length)
        ]
        frame_tensor=torch.stack(frames)
        flow_tensor =torch.stack(flows)
        if self.training:
            flow_tensor=augment_flow(flow_tensor)

        density_maps=[]
        for t in range(self.seq_length):
            d_path=os.path.join(seq_path,f"density_{start+t:03d}.npy")
            d=(np.load(d_path).astype(np.float32) if os.path.exists(d_path)
               else np.zeros((DENSITY_RES,DENSITY_RES),dtype=np.float32))
            density_maps.append(torch.tensor(d).unsqueeze(0))

        return (frame_tensor, flow_tensor,
                gt_per_frame,                # (T,) per-frame GT counts
                torch.stack(density_maps))   # (T,1,28,28)


# ============================================================
# CELL 3 : MODEL
# ============================================================

class ConvLSTMCell(nn.Module):
    def __init__(self,input_dim,hidden_dim,kernel_size=3):
        super().__init__(); self.hidden_dim=hidden_dim; p=kernel_size//2
        self.conv=nn.Conv2d(input_dim+hidden_dim,4*hidden_dim,kernel_size,padding=p)
        self.bn=nn.BatchNorm2d(4*hidden_dim)
    def forward(self,x,states):
        h,c=states; gates=self.bn(self.conv(torch.cat([x,h],dim=1)))
        i,f,o,g=torch.chunk(gates,4,dim=1)
        i=torch.sigmoid(i);f=torch.sigmoid(f);o=torch.sigmoid(o);g=torch.tanh(g)
        c_next=f*c+i*g; return o*torch.tanh(c_next),c_next

class SEBlock(nn.Module):
    def __init__(self,channels,reduction=4):
        super().__init__(); self.pool=nn.AdaptiveAvgPool2d(1)
        self.fc=nn.Sequential(
            nn.Linear(channels,max(channels//reduction,4)),nn.ReLU(),
            nn.Linear(max(channels//reduction,4),channels),nn.Sigmoid())
    def forward(self,x):
        B,C,_,_=x.shape; return x*self.fc(self.pool(x).view(B,C)).view(B,C,1,1)

class MotionResBlock(nn.Module):
    def __init__(self,in_ch,out_ch,stride=1):
        super().__init__(); mid=out_ch*2
        self.pw1=nn.Sequential(nn.Conv2d(in_ch,mid,1,bias=False),nn.BatchNorm2d(mid),nn.Hardswish())
        self.dw =nn.Sequential(nn.Conv2d(mid,mid,3,stride=stride,padding=1,groups=mid,bias=False),nn.BatchNorm2d(mid),nn.Hardswish())
        self.se =SEBlock(mid)
        self.pw2=nn.Sequential(nn.Conv2d(mid,out_ch,1,bias=False),nn.BatchNorm2d(out_ch))
        self.res=(stride==1 and in_ch==out_ch)
    def forward(self,x):
        out=self.pw2(self.se(self.dw(self.pw1(x)))); return out+x if self.res else out

class MotionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem  =nn.Sequential(nn.Conv2d(2,16,3,stride=2,padding=1,bias=False),nn.BatchNorm2d(16),nn.Hardswish())
        self.blocks=nn.Sequential(
            MotionResBlock(16,24,stride=2),MotionResBlock(24,32,stride=2),
            MotionResBlock(32,48,stride=1),MotionResBlock(48,64,stride=2),
            MotionResBlock(64,64,stride=1))
        self.final =nn.Sequential(nn.Conv2d(64,64,1,bias=False),nn.BatchNorm2d(64),nn.Hardswish(),nn.AdaptiveAvgPool2d((7,7)))
        self.drop  =nn.Dropout2d(0.15)
    def forward(self,x): return self.drop(self.final(self.blocks(self.stem(x))))


class CrowdNet(nn.Module):
    def __init__(self):
        super().__init__(); HIDDEN=128; self.hidden_dim=HIDDEN

        # Weight loading priority:
        # 1. If best_crowd_model.pth exists → backbone will be overwritten
        #    by synthetic weights in run_training(). Load ImageNet only as
        #    a fallback for modules NOT in the synthetic checkpoint.
        # 2. If best_crowd_model.pth does NOT exist → ImageNet weights are
        #    the best available starting point.
        # Either way we always need a fully constructed MobileNetV3 here.
        if os.path.exists("mobilenet_v3_small-047dcff4.pth"):
            print("BACKBONE: local mobilenet_v3_small-047dcff4.pth")
            mobilenet=models.mobilenet_v3_small(weights=None)
            mobilenet.load_state_dict(torch.load("mobilenet_v3_small-047dcff4.pth",
                                                  weights_only=True))
        else:
            print("BACKBONE: ImageNet pretrained (torchvision)")
            mobilenet=models.mobilenet_v3_small(
                weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        # Note: if best_crowd_model.pth is present (normal case), these
        # ImageNet weights will be immediately overwritten by synthetic
        # crowd-trained weights in run_training() → load_state_dict().
        # The ImageNet step here only matters if best_crowd_model.pth
        # is missing, in which case ImageNet is the best fallback.

        features=mobilenet.features
        features=mobilenet.features
        # Classification path — FULLY FROZEN throughout DroneCrowd training
        self.backbone_stage1=features[:3]    # (B,24,28,28) frozen
        self.backbone_stage2=features[3:]    # (B,576,7,7)  frozen
        for p in self.backbone_stage1.parameters(): p.requires_grad=False
        for p in self.backbone_stage2.parameters(): p.requires_grad=False

        # ── THREE-SCALE DENSITY PATH ───────────────────────────────
        # MobileNetV3-Small split points at 448×448 input:
        #   features[:3]  → (B,24,56,56)  fine spatial
        #   features[3:9] → (B,48,28,28)  intermediate
        #   features[9:]  → (B,576,14,14) semantic
        #
        # FPN top-down: stage2(7)→upsample→concat(mid,14)→upsample→concat(s1,28)
        # ASPP-lite: 4 parallel dilated convs at 28×28 (CSRNet design)
        #   rate 1,2,4,6 → captures persons, clusters, groups, crowd-regions
        # Task B: classification uses ORIGINAL frozen backbone — fully isolated.
        import copy
        self.d_stage1 = copy.deepcopy(features[:3])   # (B,24,28,28)
        self.d_mid    = copy.deepcopy(features[3:9])  # (B,48,14,14)
        self.d_stage2 = copy.deepcopy(features[9:])   # (B,576,7,7)
        for p in self.d_stage1.parameters(): p.requires_grad=True
        for p in self.d_mid.parameters():    p.requires_grad=True
        for p in self.d_stage2.parameters(): p.requires_grad=True

        # FPN lateral projections — no BN (CSRNet style):
        # backbone already normalises; BN here interferes with density scale.
        self.fpn_s2  = nn.Sequential(nn.Conv2d(576, 64, 1), nn.ReLU())
        self.fpn_mid = nn.Sequential(nn.Conv2d(112, 64, 1), nn.ReLU())
        FPN_OUT = 64 + 24  # concat(stage2_top-down, stage1) at 56×56

        # ASPP-lite: 4 parallel dilated branches — no BN (CSRNet style)
        ASPP_CH = 64
        self.aspp1 = nn.Sequential(nn.Conv2d(FPN_OUT,ASPP_CH,1),nn.ReLU())
        self.aspp2 = nn.Sequential(nn.Conv2d(FPN_OUT,ASPP_CH,3,padding=2,dilation=2),nn.ReLU())
        self.aspp3 = nn.Sequential(nn.Conv2d(FPN_OUT,ASPP_CH,3,padding=4,dilation=4),nn.ReLU())
        self.aspp4 = nn.Sequential(nn.Conv2d(FPN_OUT,ASPP_CH,3,padding=6,dilation=6),nn.ReLU())
        self.density_project = nn.Sequential(
            nn.Conv2d(ASPP_CH*4, 64, 1), nn.ReLU(),
            nn.Conv2d(64, 1, 1), nn.ReLU())   # final ReLU: density ≥ 0

        self.spatial_reduce=nn.Sequential(
            nn.Conv2d(576,64,1),nn.BatchNorm2d(64),nn.ReLU(),nn.Dropout2d(0.2),
            nn.AdaptiveAvgPool2d((7,7)))   # resolution-invariant: always 7×7
            # At 224px input backbone_stage2 was already 7×7.
            # At 448px it becomes 14×14 — this pool brings it back to 7×7
            # so ConvLSTM/classifier see the same spatial size as during
            # synthetic training, preserving frozen weight compatibility.
        self.motion_cnn=MotionCNN()
        self.convlstm  =ConvLSTMCell(128,HIDDEN)
        self.pool      =nn.AdaptiveAvgPool2d(1)
        self.classifier=nn.Sequential(
            nn.Linear(HIDDEN,128),nn.ReLU(),nn.Dropout(0.5),
            nn.Linear(128,64),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(64,3))

    def forward(self, frames, flows, density_only=False):
        B,T,_,H,W=frames.shape; h=c=None; density_maps=[]
        for t in range(T):
            frame_rgb = frames[:,t].repeat(1,3,1,1)

            # ── DENSITY: 3-scale FPN + ASPP ──────────────────────
            s1 = self.d_stage1(frame_rgb)              # (B,24,56,56) at 448px
            sm = self.d_mid(s1)                        # (B,48,28,28)
            s2 = self.d_stage2(sm)                     # (B,576,14,14)
            p2 = self.fpn_s2(s2)                       # (B,64,14,14)
            p2u= F.interpolate(p2,size=(28,28),mode='bilinear',align_corners=False)
            pm = self.fpn_mid(torch.cat([p2u,sm],dim=1))  # (B,64,28,28)
            pmu= F.interpolate(pm,size=(56,56),mode='bilinear',align_corners=False)
            fused_d = torch.cat([pmu,s1],dim=1)        # (B,88,56,56)
            a1=self.aspp1(fused_d); a2=self.aspp2(fused_d)
            a3=self.aspp3(fused_d); a4=self.aspp4(fused_d)
            density_maps.append(self.density_project(torch.cat([a1,a2,a3,a4],dim=1)))

            # ── CLASSIFICATION: fully frozen ──────────────────────
            # Skipped during density training (density_only=True) for
            # ~40% speedup. Frozen weights are never updated anyway.
            if not density_only:
                s1c = self.backbone_stage1(frame_rgb)
                s2c = self.backbone_stage2(s1c)
                sp  = self.spatial_reduce(s2c)
                fl  = self.motion_cnn(flows[:,t])
                fused_c = torch.cat([sp,fl],dim=1)
                if h is None:
                    _,_,FH,FW=fused_c.shape
                    h=torch.zeros(B,self.hidden_dim,FH,FW,device=fused_c.device)
                    c=torch.zeros_like(h)
                h,c=self.convlstm(fused_c,(h,c))

        density_stack=torch.stack(density_maps,dim=1)   # (B,T,1,56,56)
        if density_only:
            return torch.zeros(B,3,device=density_stack.device), density_stack
        logits=self.classifier(self.pool(h).view(B,-1))
        return logits,density_stack


# ============================================================
# CELL 4 : OPTIMIZER
# ============================================================

def build_optimizer(model):
    """
    Differential LR for three-scale density path:
      ASPP + FPN heads  LR=3e-4   random init
      d_stage1          LR=3e-5   fine spatial, gentle
      d_mid             LR=3e-5   intermediate, gentle
      d_stage2          LR=1e-5   semantic, very gentle (1.3M params)
    Classification: ALL FROZEN
    """
    for m in [model.backbone_stage1, model.backbone_stage2,
              model.spatial_reduce, model.motion_cnn,
              model.convlstm, model.classifier]:
        for p in m.parameters(): p.requires_grad=False
    for p in model.d_stage1.parameters(): p.requires_grad=True
    for p in model.d_mid.parameters():    p.requires_grad=True
    for p in model.d_stage2.parameters(): p.requires_grad=True

    aspp_params = (list(model.aspp1.parameters())+list(model.aspp2.parameters())+
                   list(model.aspp3.parameters())+list(model.aspp4.parameters())+
                   list(model.density_project.parameters())+
                   list(model.fpn_s2.parameters())+list(model.fpn_mid.parameters()))
    param_groups=[
        {"params": aspp_params,
         "lr": LR,       "name": "aspp+fpn+project"},
        {"params": list(model.d_stage1.parameters()),
         "lr": LR*0.1,   "name": "d_stage1"},
        {"params": list(model.d_mid.parameters()),
         "lr": LR*0.1,   "name": "d_mid"},
        {"params": list(model.d_stage2.parameters()),
         "lr": LR*0.03,  "name": "d_stage2"},
    ]
    for g in param_groups:
        n=sum(p.numel() for p in g["params"])
        print(f"  {g['name']:<20} LR={g['lr']:.0e}  params={n:,}")

    opt  =optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    sched=optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    return opt, sched


# ============================================================
# CELL 5 : TRAIN / VAL LOOPS
# ============================================================

def train_epoch(model, loader, opt, epoch):
    aug_prob=0.3 if epoch<=5 else (0.5 if epoch<=15 else 0.7)
    loader.dataset.aug_prob=aug_prob
    model.train()
    # Trainable density modules → train mode
    model.d_stage1.train()
    model.d_mid.train()
    model.d_stage2.train()
    model.aspp1.train(); model.aspp2.train()
    model.aspp3.train(); model.aspp4.train()
    model.density_project.train()
    model.fpn_s2.train(); model.fpn_mid.train()
    # Frozen classification modules → eval mode (stable BN stats)
    model.backbone_stage1.eval()
    model.backbone_stage2.eval()
    model.spatial_reduce.eval()
    model.motion_cnn.eval()
    model.convlstm.eval()
    model.classifier.eval()

    den_crit=nn.SmoothL1Loss(beta=0.1)
    total_loss=total_mae=0.0; n_batches=0; n_frames=0
    loop=tqdm(enumerate(loader),total=len(loader),leave=True)

    for bi,(frames,flows,gt_per_frame,density_stack) in loop:
        frames=frames.to(device); flows=flows.to(device)
        density_stack=density_stack.to(device)
        gt_per_frame=gt_per_frame.to(device)   # (B,T) per-frame GT counts
        opt.zero_grad()
        _,pred_stack=model(frames,flows,density_only=True)

        # Pixel-level density loss (unchanged)
        density_loss=den_crit(pred_stack, density_stack)

        # Per-frame count loss: pred count at EACH frame vs GT count at EACH frame
        # NOT the window mean — fixes the metric bug where predicting a constant
        # mean count scored zero error regardless of temporal variation
        pred_per_frame=pred_stack[:,:,0,:,:].sum(dim=[-1,-2])  # (B,T)
        count_loss=F.l1_loss(pred_per_frame, gt_per_frame)      # per-frame L1

        loss=density_loss + COUNT_LOSS_WEIGHT * count_loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        opt.step()

        # Per-frame MAE: flatten (B,T) → (B*T,)
        mae_sum=torch.abs(pred_per_frame-gt_per_frame).sum().item()
        total_loss+=loss.item(); total_mae+=mae_sum
        n_batches+=1; n_frames+=pred_per_frame.numel()
        loop.set_description(
            f"TRAIN ep{epoch} aug={aug_prob:.1f} | "
            f"Loss {total_loss/n_batches:.4f} | MAE {total_mae/n_frames:.2f}")

    return total_loss/n_batches, total_mae/n_frames


def val_epoch(model, loader):
    model.eval()
    den_crit=nn.SmoothL1Loss(beta=0.1)
    total_loss=mae_sum=mse_sum=0.0; nb=ns=0
    with torch.no_grad():
        for frames,flows,gt_per_frame,density_stack in tqdm(loader,desc="VAL"):
            frames=frames.to(device); flows=flows.to(device)
            density_stack=density_stack.to(device)
            gt_per_frame=gt_per_frame.to(device)      # (B,T)
            _,pred_stack=model(frames,flows,density_only=True)
            density_loss=den_crit(pred_stack,density_stack)
            pred_per_frame=pred_stack[:,:,0,:,:].sum(dim=[-1,-2])  # (B,T)
            count_loss=F.l1_loss(pred_per_frame, gt_per_frame)
            loss=density_loss + COUNT_LOSS_WEIGHT * count_loss
            # Per-frame MAE: flatten B×T → B*T individual frame predictions
            diff=(pred_per_frame-gt_per_frame).view(-1)
            mae_sum+=torch.abs(diff).sum().item()
            mse_sum+=(diff**2).sum().item()
            total_loss+=loss.item(); nb+=1
            ns+=diff.numel()   # count individual frames, not windows

    return total_loss/nb, mae_sum/ns, math.sqrt(mse_sum/ns)


# ============================================================
# CELL 6 : TRAINING DRIVER
# ============================================================

def run_training(processed_root, use_synthetic=True):
    """
    use_synthetic=True  → load best_crowd_model.pth (synthetic pre-training)
                          into both classification and density backbone clones.
    use_synthetic=False → start from ImageNet weights only (loaded in
                          CrowdNet.__init__). Classification branch still
                          uses ImageNet weights (frozen). Density backbone
                          clones also start from ImageNet — no synthetic data.

    Checkpoint saved as:
      use_synthetic=True  → best_dronecrowd_model.pth
      use_synthetic=False → best_dronecrowd_imagenet.pth
    So both runs can coexist and be compared.
    """
    run_label  = "SYNTHETIC"   if use_synthetic else "IMAGENET-ONLY"
    ckpt_name  = "best_dronecrowd_model.pth" if use_synthetic \
                 else "best_dronecrowd_imagenet.pth"

    print(f"\n  Weight initialisation : {run_label}")
    print(f"  Checkpoint save path  : {ckpt_name}")

    train_ds=DroneCrowdDataset(processed_root,"train",training=True,aug_prob=0.3)
    val_ds  =DroneCrowdDataset(processed_root,"val",  training=False)

    print(f"\n  Split summary:")
    print(f"  Train : {len(train_ds.windows):5d} windows (train_data, all 82 seqs, T={SEQ_LENGTH})")
    print(f"  Val   : {len(val_ds.windows):5d} windows (val_data, 30 seqs × 1 window, T={DroneCrowdDataset.VAL_SEQ_LENGTH})")
    print(f"  Test  : test_data, 30 seqs — used only for final Task A/B/C, never during training")

    train_loader=DataLoader(train_ds,BATCH_SIZE,shuffle=True, num_workers=0,pin_memory=True)
    val_loader  =DataLoader(val_ds,  BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True)

    model=CrowdNet().to(device)
    # CrowdNet.__init__ already loaded ImageNet (or local .pth) weights
    # into backbone_stage1/2 and cloned them into d_stage1/d_mid/d_stage2.

    SYNTH="best_crowd_model.pth"
    if use_synthetic:
        if os.path.exists(SYNTH):
            ckpt=torch.load(SYNTH,weights_only=True)
            # Exclude density head keys — different architecture shape
            ckpt_filtered={k:v for k,v in ckpt.items()
                           if not k.startswith("density_head")}
            missing,unexpected=model.load_state_dict(ckpt_filtered,strict=False)
            print(f"\nSYNTHETIC WEIGHTS: loaded {len(ckpt_filtered)} keys, "
                  f"skipped {len(ckpt)-len(ckpt_filtered)} density_head keys")
            print(f"  Missing={len(missing)}  Unexpected={len(unexpected)}")
            # d_stage1/d_mid/d_stage2 were cloned BEFORE synthetic load above.
            # Re-sync them so the density path also starts from synthetic weights.
            model.d_stage1.load_state_dict(model.backbone_stage1.state_dict())
            s2_sd       = model.backbone_stage2.state_dict()
            mid_keys    = [k for k in s2_sd if int(k.split('.')[0]) < 6]
            stage2_keys = [k for k in s2_sd if int(k.split('.')[0]) >= 6]
            mid_sd  = {k: s2_sd[k] for k in mid_keys}
            s2_only = {str(int(k.split('.')[0])-6)+'.'+'.'.join(k.split('.')[1:]):
                       s2_sd[k] for k in stage2_keys}
            try:
                model.d_mid.load_state_dict(mid_sd, strict=False)
                model.d_stage2.load_state_dict(s2_only, strict=False)
            except Exception as e:
                print(f"  Note: density backbone partial sync ({e})")
            print(f"  d_stage1/d_mid/d_stage2 re-synced to synthetic weights")
        else:
            print(f"\nWARNING: {SYNTH} not found — falling back to ImageNet weights.")
            print(f"  (Run will proceed as ImageNet-only)")
    else:
        # ImageNet-only run: d_stage1/d_mid/d_stage2 already hold ImageNet
        # weights from __init__ clone — nothing extra to load.
        # Classification branch (backbone_stage1/2, spatial_reduce,
        # motion_cnn, convlstm, classifier) also starts from ImageNet.
        # All classification modules are frozen, so Task B will reflect
        # ImageNet-initialised (not crowd-trained) classification features.
        print(f"\nIMAGENET-ONLY: using ImageNet backbone weights.")
        print(f"  density path (d_stage1/d_mid/d_stage2): ImageNet init")
        print(f"  classification path (frozen): ImageNet init")
        print(f"  Note: Task B classification quality will differ from")
        print(f"  synthetic run — classification branch was not crowd-trained.")

    print("\nOptimizer:")
    opt,sched=build_optimizer(model)

    best_mae=1e9
    for epoch in range(1,EPOCHS+1):
        print(f"\n{'='*60}\nEPOCH {epoch}/{EPOCHS}  [{run_label}]\n{'='*60}")
        tr_loss,tr_mae=train_epoch(model,train_loader,opt,epoch)
        vl_loss,vl_mae,vl_mse=val_epoch(model,val_loader)
        sched.step()
        print(f"TRAIN | Loss {tr_loss:.4f} | MAE {tr_mae:.2f}")
        print(f"VAL   | Loss {vl_loss:.4f} | MAE {vl_mae:.2f} | MSE {vl_mse:.2f}")
        print(f"LR    : {opt.param_groups[0]['lr']:.6f}")
        if vl_mae < best_mae:
            best_mae=vl_mae
            torch.save(model.state_dict(), ckpt_name)
            print(f"MODEL IMPROVED → SAVED  (val MAE={vl_mae:.2f})  → {ckpt_name}")
        else:
            print(f"no improvement  (best so far: {best_mae:.2f})")

    print(f"\nBest val MAE [{run_label}]: {best_mae:.2f}")
    return model, ckpt_name


# ============================================================
# CELL 7 : TASK A — DENSITY EVALUATION
# ============================================================

def evaluate_density(model, processed_root):
    """
    Evaluates on ALL 30 test sequences (no leakage).
    Correct DroneCrowd per-frame MAE: for every frame t in every window,
    compare predicted_density[t].sum() vs GT_count[t] independently.
    Final MAE = total_abs_error / total_frame_count.
    """
    ds    =DroneCrowdDataset(processed_root,"test_data",training=False)
    loader=DataLoader(ds,BATCH_SIZE,shuffle=False,num_workers=0)
    model.eval()
    mae_sum=mse_sum=0.0; n_frames=0
    with torch.no_grad():
        for frames,flows,gt_per_frame,_ in tqdm(loader,desc="EVAL"):
            _,pred_stack=model(frames.to(device),flows.to(device),density_only=True)
            # Per-frame predicted counts (B,T) and GT counts (B,T)
            pred=pred_stack[:,:,0,:,:].sum(dim=[-1,-2]).cpu().numpy()  # (B,T)
            gt  =gt_per_frame.numpy()                                    # (B,T)
            diff=pred-gt
            mae_sum+=np.abs(diff).sum()
            mse_sum+=(diff**2).sum()
            n_frames+=diff.size   # B*T individual frame comparisons
    mae=mae_sum/n_frames; mse=math.sqrt(mse_sum/n_frames)
    print("\n"+"="*65)
    print("TASK A — CROWD COUNTING (all 30 test seqs, per-frame MAE)")
    print("="*65)
    print(f"\n  {'Method':<38} {'MAE':>8}  {'MSE':>8}")
    print("  "+"-"*58)
    for method,m,s in [
        ("STNNet  (Wen, CVPR 2021)",  25.14,43.04),
        ("STANet  (Wen, arXiv 2021)", 21.15,35.19),
        ("DenseTrack (2024)",          18.31,29.68),
        ("CrowdNet (ours)",            mae,  mse),
    ]:
        arrow=" ←" if "ours" in method else ""
        print(f"  {method:<38} {m:>8.2f}  {s:>8.2f}{arrow}")
    print("="*65)
    return mae,mse


# ============================================================
# CELL 8 : TASK B — QUALITATIVE BEHAVIOUR CLASSIFICATION
# ============================================================

def qualitative_classification(model, processed_root, run_label="SYNTHETIC"):
    """
    Runs the frozen classification branch on all 30 test sequences.
    The classification weights reflect whatever initialisation was used:
      run_label="SYNTHETIC"     → crowd-trained synthetic weights (frozen)
      run_label="IMAGENET-ONLY" → ImageNet weights (frozen, no crowd training)
    Model is passed in already loaded with the best checkpoint — no reload.
    """
    model.eval()
    classif_note = ("synthetic crowd-trained" if run_label=="SYNTHETIC"
                    else "ImageNet-only (no crowd training)")
    split_root=os.path.join(processed_root,"test_data")
    seq_names =sorted([s for s in os.listdir(split_root)
                       if os.path.isdir(os.path.join(split_root,s))])
    seq_class_dist=defaultdict(lambda:{0:0,1:0,2:0})
    seq_gt_counts =defaultdict(list)

    for seq_name in seq_names:
        seq_path=os.path.join(split_root,seq_name)
        n_frames=len(glob.glob(os.path.join(seq_path,"frame_*.png")))
        gt_path =os.path.join(seq_path,"gt_counts.npy")
        gt_arr  =(np.load(gt_path) if os.path.exists(gt_path)
                  else np.zeros(n_frames,dtype=np.float32))
        ds=DroneCrowdDataset.__new__(DroneCrowdDataset)
        ds.seq_length=SEQ_LENGTH; ds.training=False; ds.aug_prob=0.0; ds.windows=[]
        for start in range(WARMUP_FRAMES,n_frames-SEQ_LENGTH,STRIDE):
            window_gt=gt_arr[start:start+SEQ_LENGTH]
            ds.windows.append({"seq_path":seq_path,"start":start,
                                "gt_counts_per_frame":window_gt.tolist()})
        if not ds.windows: continue
        loader=DataLoader(ds,batch_size=4,shuffle=False,num_workers=0)
        with torch.no_grad():
            for frames,flows,gt_pf,_ in loader:
                logits,_=model(frames.to(device),flows.to(device))
                for p in torch.argmax(logits,dim=1).cpu().numpy():
                    seq_class_dist[seq_name][int(p)]+=1
                seq_gt_counts[seq_name].extend(gt_pf.mean(dim=1).tolist())

    print("\n"+"="*75)
    print(f"TASK B — QUALITATIVE BEHAVIOUR CLASSIFICATION [{run_label}]")
    print(f"  Classification weights: {classif_note}")
    print("="*75)
    print(f"{'Sequence':<12} {'AvgCount':>9} {'Attr':>8}  "
          f"{'Normal%':>8} {'Bottl%':>7} {'Panic%':>7}  {'Dominant'}")
    print("-"*75)
    for seq_name in sorted(seq_class_dist.keys()):
        dist=seq_class_dist[seq_name]; total_w=max(sum(dist.values()),1)
        avg_count=np.mean(seq_gt_counts[seq_name]) if seq_gt_counts[seq_name] else 0.0
        attr="Crowded" if avg_count>CROWDED_THRESH else "Sparse"
        dominant=CLASS_NAMES[max(dist,key=dist.get)]
        pct={c:100*dist[c]/total_w for c in range(3)}
        print(f"{seq_name:<12} {avg_count:>9.1f} {attr:>8}  "
              f"{pct[0]:>7.1f}% {pct[1]:>6.1f}% {pct[2]:>6.1f}%  {dominant}")
    print("="*75)
    return seq_class_dist



# ============================================================
# CELL 9 : TASK C — MotionCNN GRADIENT vs GT TRAJECTORIES
# ============================================================
#
# Correct methodology: Test whether MotionCNN's internal feature
# response encodes motion direction information from GT trajectories.
#
# Previous approach (v7): patch-averaged Farneback flow compared to
# GT displacement. This tested the RAW FLOW INPUT, not MotionCNN.
# The user correctly identified this as measuring the wrong thing.
#
# Correct approach (v8): patch-averaged MotionCNN GRADIENT.
# For each person at (px, py) with GT displacement (dx, dy):
#   1. Run the real Farneback flow through MotionCNN
#   2. Get activation at the person's 7×7 cell: feat[0,:,cy,cx].sum()
#   3. Backpropagate to get gradient w.r.t. flow input
#   4. Average the gradient over a patch sized to the receptive field
#      (~79×79px at 224px scale) — this gives the MEAN direction
#      the network is most sensitive to around that person
#   5. Compare this mean sensitivity direction to GT displacement
#
# Previous single-pixel gradient failed because one pixel's gradient
# contribution across a 79×79 receptive field is noise-dominated.
# Patch-averaging recovers the aggregate directional sensitivity.
#
# At 448×448 input: receptive field is 2× larger (~158×158px).
# We scale the patch radius accordingly.
#

def motioncnn_patch_gradient_direction(motion_cnn, flow_np, px, py,
                                        model_size=MODEL_SIZE):
    """
    Computes the patch-averaged gradient direction of MotionCNN at (px,py).
    
    flow_np : (H, W, 2) float32 — Farneback flow, clipped to [-1,1]
    Returns:
        grad_dir : (2,) float32 — mean gradient direction in patch
        act_val  : float        — activation magnitude at (cx,cy)
    """
    flow_t = torch.tensor(flow_np, dtype=torch.float32).permute(2,0,1).unsqueeze(0).to(device)
    flow_t.requires_grad_(True)

    # Forward through MotionCNN
    feat = motion_cnn(flow_t)          # (1, 64, 7, 7)
    n_cells = feat.shape[-1]           # 7 (or 14 at 448px — MotionCNN has fixed 7×7)

    # Map person pixel → MotionCNN output cell
    cx = int(np.clip(px / model_size * n_cells, 0, n_cells-1))
    cy = int(np.clip(py / model_size * n_cells, 0, n_cells-1))

    # Scalar activation at this cell (sum over channels)
    act_val = feat[0, :, cy, cx].sum()
    act_val.backward()

    if flow_t.grad is None:
        return np.zeros(2, dtype=np.float32), 0.0

    grad = flow_t.grad[0]  # (2, H, W)

    # Receptive field of MotionCNN at 7×7 output ≈ 79px at 224px input.
    # At 448px input (MODEL_SIZE=448), the same network architecture gives
    # RF ≈ 158px. Use half of RF as patch radius.
    rf_half = max(1, int(model_size / 224 * 39))   # scales with resolution
    y0 = max(0, py - rf_half); y1 = min(grad.shape[1], py + rf_half + 1)
    x0 = max(0, px - rf_half); x1 = min(grad.shape[2], px + rf_half + 1)

    # Average gradient over the receptive field patch: (2,)
    patch_grad = grad[:, y0:y1, x0:x1].mean(dim=[1,2]).detach().cpu().numpy()
    return patch_grad.astype(np.float32), float(act_val.item())


def compare_motioncnn_to_trajectories(model, processed_root,
                                       max_seqs=TASK_C_MAX_SEQS,
                                       max_frames=TASK_C_MAX_FRAMES):
    """
    Compares MotionCNN's patch-averaged gradient direction to GT trajectory
    displacement direction. Tests whether MotionCNN encodes directional
    motion information from the crowd flow field.
    
    Signal above random chance → MotionCNN has learned directional motion.
    Signal at random chance → MotionCNN is direction-agnostic.
    """
    model.eval()
    motion_cnn = model.motion_cnn
    split_root = os.path.join(processed_root, "test_data")
    seq_names  = sorted([s for s in os.listdir(split_root)
                         if os.path.isdir(os.path.join(split_root, s))])[:max_seqs]

    cos_sims=[]; angle_errs=[]; activations=[]
    skipped_stat=0; skipped_weak_grad=0; total_cands=0

    for seq_name in tqdm(seq_names, desc="Task C"):
        seq_path  = os.path.join(split_root, seq_name)
        traj_path = os.path.join(seq_path, "trajectories.npy")
        if not os.path.exists(traj_path):
            print(f"  {seq_name}: no trajectories.npy"); continue

        traj        = np.load(traj_path)   # (M,4): t,x_224,y_224,track_id
        frame_files = sorted(glob.glob(os.path.join(seq_path, "frame_*.png")))
        nf          = min(len(frame_files)-1, max_frames)

        frame_pos = defaultdict(dict)
        for row in traj:
            t=int(row[0]); frame_pos[t][int(row[3])]=(row[1],row[2])

        for t in range(nf):
            # Load precomputed flow if available, else compute on the fly
            flow_path = os.path.join(seq_path, f"flow_{t:03d}.npy")
            if os.path.exists(flow_path):
                flow_np = np.load(flow_path).astype(np.float32)
            else:
                img_t  = cv2.imread(frame_files[t],   cv2.IMREAD_GRAYSCALE)
                img_t1 = cv2.imread(frame_files[t+1], cv2.IMREAD_GRAYSCALE)
                if img_t is None or img_t1 is None: continue
                flow_np = np.clip(cv2.calcOpticalFlowFarneback(
                    img_t, img_t1, None, pyr_scale=0.5, levels=3, winsize=15,
                    iterations=3, poly_n=5, poly_sigma=1.2, flags=0)/8.0, -1.0, 1.0)

            pos_t  = frame_pos.get(t,   {})
            pos_t1 = frame_pos.get(t+1, {})
            common = set(pos_t) & set(pos_t1)
            total_cands += len(common)

            for tid in common:
                x0,y0 = pos_t[tid];  x1,y1 = pos_t1[tid]
                gt_vec = np.array([x1-x0, y1-y0], dtype=np.float64)
                gt_mag = np.linalg.norm(gt_vec)

                # Skip near-stationary persons (< 0.1px in MODEL_SIZE space)
                if gt_mag < 0.1:
                    skipped_stat += 1; continue

                px_ = int(np.clip(x0, 0, MODEL_SIZE-1))
                py_ = int(np.clip(y0, 0, MODEL_SIZE-1))

                # Patch-averaged MotionCNN gradient direction
                grad_dir, act = motioncnn_patch_gradient_direction(
                    motion_cnn, flow_np, px_, py_, model_size=MODEL_SIZE)

                grad_mag = np.linalg.norm(grad_dir)
                if grad_mag < 1e-4:
                    skipped_weak_grad += 1; continue

                cosine = float(np.clip(
                    np.dot(gt_vec, grad_dir.astype(np.float64)) /
                    ((gt_mag + 1e-6) * grad_mag), -1, 1))
                cos_sims.append(cosine)
                angle_errs.append(float(np.degrees(np.arccos(cosine))))
                activations.append(act)

    print(f"\n  Total candidates       : {total_cands:,}")
    print(f"  Skipped (stationary)   : {skipped_stat:,}")
    print(f"  Skipped (weak gradient): {skipped_weak_grad:,}")
    print(f"  Evaluated              : {len(cos_sims):,}")

    if not cos_sims:
        print("No evaluable persons found.")
        return {}

    cos_sims   = np.array(cos_sims)
    angle_errs = np.array(angle_errs)
    activations= np.array(activations)

    # Random baseline (uniform random 2D directions)
    rng = np.random.RandomState(0)
    rand_a  = rng.uniform(0, 2*np.pi, 5000)
    rand_b  = rng.uniform(0, 2*np.pi, 5000)
    rdiff   = np.abs(rand_a - rand_b)
    rdiff   = np.minimum(rdiff, 2*np.pi - rdiff)
    rand_deg = np.degrees(rdiff)

    print("\n" + "="*65)
    print("TASK C — MotionCNN GRADIENT vs GT TRAJECTORY DIRECTION")
    print("="*65)
    print(f"  Method: patch-averaged gradient (RF patch ~{int(MODEL_SIZE/224*79)}px)")
    print(f"  Persons evaluated      : {len(cos_sims):,}")
    print(f"  Mean cosine similarity : {cos_sims.mean():.4f}  (0=random, 1=perfect)")
    print(f"  Mean angle error       : {angle_errs.mean():.2f}°  "
          f"(random baseline: {rand_deg.mean():.2f}°)")
    print(f"  Median angle error     : {np.median(angle_errs):.2f}°")
    print(f"  Fraction < 15°         : {(angle_errs<15).mean()*100:.1f}%  "
          f"(random: {(rand_deg<15).mean()*100:.1f}%)")
    print(f"  Fraction < 30°         : {(angle_errs<30).mean()*100:.1f}%  "
          f"(random: {(rand_deg<30).mean()*100:.1f}%)")
    print(f"  Fraction < 45°         : {(angle_errs<45).mean()*100:.1f}%")
    print(f"  Mean MotionCNN activation: {activations.mean():.4f}")

    improvement = rand_deg.mean() - angle_errs.mean()
    sig = improvement > 5.0
    print(f"\n  Angle improvement over random: {improvement:.2f}°")
    print(f"  Verdict: {'DIRECTIONAL SIGNAL DETECTED ✓' if sig else 'NO CLEAR SIGNAL ✗ — MotionCNN may be direction-agnostic'}")

    bins=[0,15,30,45,60,90,135,180]
    lbls=["0-15°","15-30°","30-45°","45-60°","60-90°","90-135°","135-180°"]
    cnts,_ = np.histogram(angle_errs, bins=bins)
    total  = len(angle_errs)
    print("\n  Angle error distribution:")
    for lbl, cnt in zip(lbls, cnts):
        print(f"    {lbl:>10}  {cnt:>6}  ({100*cnt/total:5.1f}%)  "
              f"{'█'*int(cnt/max(cnts)*30)}")
    print("="*65)
    return {"mean_cosine":  float(cos_sims.mean()),
            "mean_angle":   float(angle_errs.mean()),
            "pct_lt_15":    float((angle_errs<15).mean()*100),
            "random_angle": float(rand_deg.mean()),
            "improvement":  float(improvement)}

# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print(f"\nRUN MODE: MAX_SEQS_SMOKE = {MAX_SEQS_SMOKE}\n")

    # ── SKIP_TRAINING flags ───────────────────────────────────────────────
    # Set True for a run whose checkpoint already exists, to skip training
    # and go straight to evaluation (Task A/B/C).
    #
    #   SYNTHETIC     saves → best_dronecrowd_model.pth
    #   IMAGENET-ONLY saves → best_dronecrowd_imagenet.pth
    #
    # Example — synthetic already done, run imagenet-only fresh:
    #   SKIP_TRAINING = {"SYNTHETIC": True, "IMAGENET-ONLY": False}
    #
    SKIP_TRAINING = {
        "SYNTHETIC":     True,
        "IMAGENET-ONLY": False,
    }
    # ─────────────────────────────────────────────────────────────────────

    # Preprocessing: always called — per-sequence skip logic is built-in.
    # Completed sequences are detected automatically (frames+flows+traj
    # all present) and skipped in seconds. No flag needed.
    print("="*60+"\nSTAGE 1 : PRE-PROCESSING\n"+"="*60)
    preprocess_all(DRONECROWD_ROOT, PROCESSED_DIR)
    print("\n"+"="*60+"\nSTAGE 1b : DENSITY SANITY CHECK\n"+"="*60)
    sanity_check_density(PROCESSED_DIR, n_frames=8)

    results = {}

    for run_idx, use_synthetic in enumerate([True, False]):
        run_label = "SYNTHETIC" if use_synthetic else "IMAGENET-ONLY"
        ckpt_name = ("best_dronecrowd_model.pth" if use_synthetic
                     else "best_dronecrowd_imagenet.pth")
        sep = "="*60

        print(f"\n\n{sep}")
        print(f"EXPERIMENT {run_idx+1}/2 : {run_label} WEIGHTS")
        print(f"{sep}")

        # ── STAGE 2: Training ─────────────────────────────────────────
        if not SKIP_TRAINING[run_label]:
            print(f"\n{sep}\nSTAGE 2 : TRAINING [{run_label}]\n{sep}")
            model, ckpt_name = run_training(PROCESSED_DIR,
                                            use_synthetic=use_synthetic)
        else:
            print(f"\nSTAGE 2 : TRAINING [{run_label}] — SKIPPED")
            print(f"  Loading existing checkpoint: {ckpt_name}")
            if not os.path.exists(ckpt_name):
                print(f"  ERROR: {ckpt_name} not found. "
                      f"Set SKIP_TRAINING['{run_label}']=False to train first.")
                continue
            model = CrowdNet().to(device)
            SYNTH = "best_crowd_model.pth"
            if use_synthetic and os.path.exists(SYNTH):
                ck   = torch.load(SYNTH, weights_only=True)
                filt = {k:v for k,v in ck.items()
                        if not k.startswith("density_head")}
                model.load_state_dict(filt, strict=False)

        # Load best checkpoint before evaluation (covers both training paths)
        model.load_state_dict(torch.load(ckpt_name, weights_only=True))
        model.eval()

        # ── STAGE 3: Task A ───────────────────────────────────────────
        print(f"\n{sep}\nSTAGE 3 : TASK A [{run_label}]\n{sep}")
        mae, mse = evaluate_density(model, PROCESSED_DIR)
        results[run_label] = {"MAE": mae, "MSE": mse}

        # ── STAGE 4: Task B ───────────────────────────────────────────
        print(f"\n{sep}\nSTAGE 4 : TASK B [{run_label}]\n{sep}")
        qualitative_classification(model, PROCESSED_DIR, run_label=run_label)

        # ── STAGE 5: Task C ───────────────────────────────────────────
        print(f"\n{sep}\nSTAGE 5 : TASK C [{run_label}]\n{sep}")
        compare_motioncnn_to_trajectories(model, PROCESSED_DIR,
                                          max_seqs=TASK_C_MAX_SEQS,
                                          max_frames=TASK_C_MAX_FRAMES)

        print(f"\n{sep}\nEXPERIMENT {run_idx+1}/2 COMPLETE [{run_label}]\n{sep}")

    # ── COMPARISON TABLE ──────────────────────────────────────────────
    if results:
        print("\n\n" + "="*65)
        print("FINAL COMPARISON — SYNTHETIC vs IMAGENET-ONLY")
        print("="*65)
        print(f"  {'Method':<42} {'MAE':>7}  {'MSE':>7}")
        print("  " + "-"*58)
        for method, m, s in [
            ("STNNet  (Wen, CVPR 2021)",   25.14, 43.04),
            ("STANet  (Wen, arXiv 2021)",  21.15, 35.19),
            ("DenseTrack (2024)",           18.31, 29.68),
        ]:
            print(f"  {method:<42} {m:>7.2f}  {s:>7.2f}")
        print("  " + "-"*58)
        for rl, r in results.items():
            print(f"  {'CrowdNet (' + rl + ')' :<42} {r['MAE']:>7.2f}  {r['MSE']:>7.2f}")
        print("="*65)

        if len(results) == 2:
            s_mae = results.get("SYNTHETIC",     {}).get("MAE")
            i_mae = results.get("IMAGENET-ONLY", {}).get("MAE")
            if s_mae is not None and i_mae is not None:
                diff = s_mae - i_mae
                print(f"\n  MAE difference (Synthetic − ImageNet): {diff:+.2f}")
                if abs(diff) < 1.5:
                    print("  → Negligible: synthetic pre-training adds little value.")
                elif diff > 0:
                    print("  → ImageNet-only is BETTER for density estimation.")
                else:
                    print("  → Synthetic pre-training HELPS density estimation.")

    print("\n" + "="*60 + "\nPIPELINE COMPLETE\n" + "="*60)

C:\Users\RAJESH\csrnet-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE : cuda

RUN MODE: MAX_SEQS_SMOKE = None

STAGE 1 : PRE-PROCESSING

Pre-processing: train_data
  [train_data] sample filenames: ['img001001.jpg', 'img001002.jpg', 'img001003.jpg', 'img001004.jpg']
  [train_data] found 82 sequences (24600 frames total)  [min_frames=20]


  preprocess train_data:   0%|                                                                  | 0/82 [00:00<?, ?it/s]

    seq001: done (300 frames, 299 flows ✓)
    seq002: done (300 frames, 299 flows ✓)
    seq003: done (300 frames, 299 flows ✓)
    seq004: done (300 frames, 299 flows ✓)


  preprocess train_data:  26%|██████████████▎                                         | 21/82 [00:00<00:00, 101.20it/s]

...


TASK A — CROWD COUNTING (all 30 test seqs, per-frame MAE)

  Method                                      MAE       MSE
  ----------------------------------------------------------
  STNNet  (Wen, CVPR 2021)                  25.14     43.04
  STANet  (Wen, arXiv 2021)                 21.15     35.19
  DenseTrack (2024)                         18.31     29.68
  CrowdNet (ours)                           43.14     57.56 ←

STAGE 4 : TASK B [SYNTHETIC]

TASK B — QUALITATIVE BEHAVIOUR CLASSIFICATION [SYNTHETIC]
  Classification weights: synthetic crowd-trained
Sequence      AvgCount     Attr   Normal%  Bottl%  Panic%  Dominant
---------------------------------------------------------------------------
seq011           205.6  Crowded     98.2%    1.8%    0.0%  Normal
seq015           148.5   Sparse     80.7%   19.3%    0.0%  Normal
seq016           146.3   Sparse     59.6%   40.4%    0.0%  Normal
seq017           159.4  Crowded     82.5%    1.8%   15.8%  Normal
seq018           143.0   Spars

Task C: 100%|███████████████████████████████████████████████████████████████████████████| 5/5 [09:35<00:00, 115.01s/it]



  Total candidates       : 39,440
  Skipped (stationary)   : 19,361
  Skipped (weak gradient): 8
  Evaluated              : 20,071

TASK C — MotionCNN GRADIENT vs GT TRAJECTORY DIRECTION
  Method: patch-averaged gradient (RF patch ~158px)
  Persons evaluated      : 20,071
  Mean cosine similarity : 0.0524  (0=random, 1=perfect)
  Mean angle error       : 85.57°  (random baseline: 89.61°)
  Median angle error     : 89.95°
  Fraction < 15°         : 18.5%  (random: 8.6%)
  Fraction < 30°         : 26.3%  (random: 17.0%)
  Fraction < 45°         : 26.9%
  Mean MotionCNN activation: 17.6177

  Angle improvement over random: 4.04°
  Verdict: NO CLEAR SIGNAL ✗ — MotionCNN may be direction-agnostic

  Angle error distribution:
         0-15°    3706  ( 18.5%)  █████████████████████
        15-30°    1565  (  7.8%)  █████████
        30-45°     127  (  0.6%)  
        45-60°     285  (  1.4%)  █
        60-90°    5195  ( 25.9%)  ██████████████████████████████
       90-135°    4914  ( 24.5%) 

TRAIN ep1 aug=0.3 | Loss 2.9437 | MAE 57.99:  33%|███████████▎                      | 388/1169 [19:42<39:39,  3.05s/it]


KeyboardInterrupt: 